<a href="https://colab.research.google.com/github/bfs0/SDAIA_Athkax_A-B_Testing_PROJECT/blob/main/Copy_of_%D8%AA%D9%85%D8%B1%D9%8A%D9%86_2_1%D8%8C_%D8%AD%D8%AC%D9%85_%D8%A7%D9%84%D8%B9%D9%8A%D9%86%D8%A9_%D9%88%D8%A7%D9%84%D9%85%D8%AF%D8%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# تمرين 2.1: حجم العينة والمدة

**برنامج التجارب واختبارات A/B والاستدلال السببي**، اليوم الثاني

**الخطوات:**
1. من قائمة File اختر Save a copy in Drive، واعمل على النسخة التي تُفتح.
2. شغّل الخلايا بالترتيب، ثم اكتب حلك في الخلية المخصصة.
3. اضغط Share، واجعل General access على Anyone with the link بصلاحية Viewer، ثم اضغط Copy link.
4. الصق الرابط في ملف التقييم الخاص بمجموعتك، في عمود «تمرين 2.1».

</div>

<div dir="rtl">

## الإعداد

شغّل الخلية التالية أولًا.

</div>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import html
from IPython.display import HTML, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib import ft2font
from scipy import stats

# الإصدارات الحديثة من matplotlib تعرض العربية مباشرة، والقديمة تحتاج مكتبتين صغيرتين
NATIVE_ARABIC = hasattr(ft2font, "__libraqm_version__")
if not NATIVE_ARABIC:
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
        import arabic_reshaper
        from bidi.algorithm import get_display

rng = np.random.default_rng(2026)

# ألوان البرنامج: A تركوازي، وB برتقالي، والقيمة الحقيقية كحلي، والطريقة الساذجة رمادي
TEAL, ORANGE, NAVY, GRAY = "#0E7490", "#E58A1F", "#0F2A3F", "#A9B4BC"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.prop_cycle": cycler(color=[TEAL, ORANGE, NAVY, GRAY]),
    "axes.edgecolor": "#56656F", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": "#3D4F5C", "ytick.color": "#3D4F5C",
    "axes.grid": True, "grid.color": "#D8E1E7", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def ar(text):
    """يجهز النص العربي ليظهر بالشكل والاتجاه الصحيحين داخل الرسوم."""
    text = str(text)
    return text if NATIVE_ARABIC else get_display(arabic_reshaper.reshape(text))


def say(*parts, sep=" "):
    """يطبع نصًا عربيًا من اليمين إلى اليسار."""
    text = html.escape(sep.join(str(p) for p in parts))
    display(HTML(f'<div dir="rtl" style="white-space: pre-wrap; line-height: 1.7;">{text}</div>'))

import statsmodels.formula.api as smf
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep
from statsmodels.stats.multitest import multipletests


def tidy(fit, terms=None):
    """جدول مختصر لنتيجة الانحدار: المعامل، والخطأ المعياري، وفترة الثقة، وقيمة p."""
    ci = fit.conf_int()
    out = pd.DataFrame({"المعامل": fit.params, "SE": fit.bse, "أدنى 95%": ci[0], "أعلى 95%": ci[1], "p": fit.pvalues})
    return out.loc[terms] if terms else out


say("الإعداد جاهز")

<div dir="rtl">

## الأدوات الجاهزة

هذه الدوال والبيانات من دفتر اليوم، ويحتاجها التمرين. شغّلها كما هي.

</div>

In [ ]:
def n_per_group_proportions(p_control, p_treatment, alpha=0.05, power=0.80):
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    var = p_control * (1 - p_control) + p_treatment * (1 - p_treatment)
    return int(np.ceil((z_a + z_b) ** 2 * var / (p_treatment - p_control) ** 2))


n_formula = n_per_group_proportions(0.10, 0.11)
n_sm = NormalIndPower().solve_power(effect_size=proportion_effectsize(0.11, 0.10), alpha=0.05, power=0.8)
say(f"بالمعادلة: {n_formula:,} في كل مجموعة، وبمكتبة statsmodels: {n_sm:,.0f} في كل مجموعة")

In [ ]:
def mde_proportions(p_control, n_per_group, alpha=0.05, power=0.80):
    """أصغر أثر مطلق نستطيع اكتشافه بعينة معروفة الحجم."""
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    return (z_a + z_b) * np.sqrt(2 * p_control * (1 - p_control) / n_per_group)


for n in [5_000, 15_000, 60_000]:
    m = mde_proportions(0.10, n)
    say(f"عند {n:,} في كل مجموعة: أصغر أثر قابل للاكتشاف حوالي {m * 100:.2f} نقطة ({m / 0.10:.0%} نسبيًا)")

pd.DataFrame({"أصغر أثر (نقطة مئوية)": [2.0, 1.0, 0.5],
              "n لكل مجموعة": [n_per_group_proportions(0.10, 0.10 + d / 100) for d in [2.0, 1.0, 0.5]]})

,أصغر أثر (نقطة مئوية),n لكل مجموعة
0,2.0,3839
1,1.0,14749
2,0.5,57760


In [ ]:
def duration_days(n_per_group, daily_new_users, groups=2, round_to_weeks=True):
    days = int(np.ceil(n_per_group * groups / daily_new_users))
    return int(np.ceil(days / 7) * 7) if round_to_weeks else days


say(f"المدة {duration_days(n_formula, 2_000, round_to_weeks=False)} يومًا، "
      f"وتصبح {duration_days(n_formula, 2_000)} يومًا بعد التقريب إلى أسابيع كاملة")

<div dir="rtl">

## المطلوب

منتج نسبة الاشتراك فيه حاليًا **4%**، والفريق يريد اكتشاف زيادة نسبية قدرها **10%** (أي من 4% إلى 4.4%). يدخل المنتج **3,000** مستخدم جديد كل يوم.
1. كم مستخدمًا تحتاج في كل مجموعة؟
2. كم يومًا ستستمر التجربة بعد التقريب إلى أسابيع كاملة؟
3. إذا كان الفريق لا يستطيع تشغيل التجربة أكثر من أسبوعين، فما أصغر أثر يمكنه اكتشافه؟

</div>

In [ ]:
# اكتب حلك هنا
# 1. كم مستخدماً تحتاج في كل مجموعة؟
n_formula = n_per_group_proportions(0.04, 0.044)
print("#1 :", n_formula)

# 2. كم يوماً ستستمر التجربة بعد التقريب إلى أسابيع كاملة؟
print("#2 :", duration_days(n_formula, 3000))

# 3. ما أصغر أثر يمكن اكتشافه في أسبوعين؟
print("#3 :", mde_proportions(0.04, 21_000))


#1 : 39473
#2 : 28
#3 : 0.005357655686683131


<div dir="rtl">

## التسليم

انتهيت؟ اضغط Share، ثم Copy link، والصق الرابط في ملف التقييم في عمود «تمرين 2.1».

</div>